In [ ]:
"""RAG Chain

In [ ]:
ChromaDBとLLMを使用したRAGチェーンの実装
"""

In [ ]:
import os
from typing import Dict, List

In [ ]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

In [ ]:
# ChromaDBの初期化

In [ ]:
chroma_client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="/tmp/shogi/chromadb",
))

In [ ]:
# Embeddingモデルの初期化

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# LLMの初期化（Gemini 2.5 Flash with Groq Llama 3.3 70B fallback）

In [ ]:
class LLMClient:
    def __init__(self):
        self.gemini_api_key = os.getenv("GEMINI_API_KEY")
        self.groq_api_key = os.getenv("GROQ_API_KEY")
        self.use_gemini = True

In [ ]:
    def generate(self, prompt: str) -> str:
        """LLMによる生成

In [ ]:
        Args:
            prompt: プロンプト

In [ ]:
        Returns:
            生成されたテキスト
        """
        if self.use_gemini and self.gemini_api_key:
            try:

In [ ]:
                # Gemini 2.5 Flashを使用

In [ ]:
                import google.generativeai as genai
                genai.configure(api_key=self.gemini_api_key)
                model = genai.GenerativeModel("gemini-2.5-flash")
                response = model.generate_content(prompt)
                return response.text
            except Exception as e:
                print(f"Gemini error: {e}, falling back to Groq")
                self.use_gemini = False

In [ ]:
        # Groq Llama 3.3 70B fallback

In [ ]:
        if self.groq_api_key:
            try:
                from groq import Groq
                client = Groq(api_key=self.groq_api_key)
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7,
                    max_tokens=1024,
                )
                return response.choices[0].message.content
            except Exception as e:
                print(f"Groq error: {e}")

In [ ]:
        return "LLM generation failed"

In [ ]:
llm_client = LLMClient()

In [ ]:
def retrieve_relevant_documents(
    query: str,
    collection_name: str = "positions",
    n_results: int = 5,
) -> List[Dict]:
    """ChromaDBから関連ドキュメントを取得

In [ ]:
    Args:
        query: クエリテキスト
        collection_name: コレクション名
        n_results: 取得するドキュメント数

In [ ]:
    Returns:
        関連ドキュメントリスト
    """
    query_embedding = embedding_model.encode(query).tolist()

In [ ]:
    try:
        collection = chroma_client.get_collection(collection_name)
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
        )

In [ ]:
        documents = []
        for i in range(len(results["documents"][0])):
            documents.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "distance": results["distances"][0][i],
            })

In [ ]:
        return documents
    except Exception as e:
        print(f"Retrieval error: {e}")
        return []

In [ ]:
def generate_response(query: str, context: List[Dict]) -> str:
    """コンテキストに基づいて回答を生成

In [ ]:
    Args:
        query: クエリテキスト
        context: 関連ドキュメントリスト

In [ ]:
    Returns:
        生成された回答
    """

In [ ]:
    # コンテキストのフォーマット

In [ ]:
    context_text = "\n\n".join([
        f"ドキュメント {i+1}:\n{doc['text']}\nメタデータ: {doc['metadata']}"
        for i, doc in enumerate(context)
    ])

In [ ]:
    # プロンプトの構築

In [ ]:
    prompt = f"""以下の将棋の棋譜情報を参考に、質問に答えてください。

In [ ]:
質問: {query}

In [ ]:
参考情報:
{context_text}

In [ ]:
回答:"""

In [ ]:
    return llm_client.generate(prompt)

In [ ]:
def rag_query(
    query: str,
    collection_name: str = "positions",
    n_results: int = 5,
) -> Dict:
    """RAGクエリを実行

In [ ]:
    Args:
        query: クエリテキスト
        collection_name: コレクション名
        n_results: 取得するドキュメント数

In [ ]:
    Returns:
        RAG結果（回答と参照ドキュメント）
    """

In [ ]:
    # 関連ドキュメントの取得

In [ ]:
    documents = retrieve_relevant_documents(query, collection_name, n_results)

In [ ]:
    if not documents:
        return {
            "answer": "関連する情報が見つかりませんでした。",
            "documents": [],
        }

In [ ]:
    # 回答の生成

In [ ]:
    answer = generate_response(query, documents)

In [ ]:
    return {
        "answer": answer,
        "documents": documents,
    }